# 牛津 Tutorial LLM 仿真 · 选修E2 Day 1
## 营销分析框架: 描述 + 诊断 (Descriptive/diagnostic analytics, funnel, cohort, segmentation, RFM)

### Persona Prompt (Oxford Tutorial Fellow + HBS Devil's Advocate)

You are an Oxford tutorial fellow in **marketing analytics: descriptive + diagnostic framework**
(AARRR funnel, RFM segmentation, t-test/chi-square/OLS on NSW RCT data).
**Never give direct answers.** Use **Socratic questioning** to draw the student's own reasoning out.
Act as a **HBS devil's advocate**: challenge every claim, demand evidence from NSW data,
reject vague assertions like "it's significant" or "the funnel works".
End **each turn** with a probing question.
If the student invokes a number (p value, R-squared, Cohen's d), ask **why** that number matters
and **what business decision** it supports.

> 限频: 本单元每天最多 1 次 tutorial (防依赖). 见 cell6.


## Pre-Tutorial Task (强制 Retrieval, 不做不放tutorial)

在进入 Socratic 对话前, 学生**必须先提交**一段 300 字 essay + 一段解题:

1. **Essay (300 字)**: 用 NSW RCT 营销映射数据, 解释 "描述性分析" 与 "诊断性分析" 的边界。
   具体说: `df.groupby('treat')['re78'].mean()` 是哪一层? `scipy.stats.ttest_ind` 是哪一层?
   `statsmodels.OLS(re78 ~ treat + re75 + age + educ)` 是哪一层? 为什么?
2. **解题**: 对 NSW 数据跑一次 t 检验 (treated vs control 的 re78), 写出 t 值/p 值/Cohen's d,
   并用一句话给出营销建议 (是否值得放量).

> 提交后 tutor 才会进入 cell3 的 Socratic loop. 这是 retrieval practice, 优于重读.


In [ ]:
# === cell3: Multi-turn Socratic Loop (STATIC if/else simulation, NO API calls) ===
# 5 苏格拉底问: 为什么/反例/若前提变/凭什么/如何
# 每轮根据学生回答分支, 模拟 Oxford tutor 追问

# 学生 pre-tutorial 提交 (模拟, 实际从 cell2 读取)
student_submission = {
    "essay_summary": "groupby 是描述性, ttest 是诊断性, OLS 也是诊断性",
    "t_value": 1.98,
    "p_value": 0.048,
    "cohens_d": 0.11,
    "recommendation": "p<0.05 显著, 建议放量"
}

def socratic_turn(turn_id, student_input):
    """静态 if/else 模拟 Socratic 追问, 不调 openai/anthropic"""
    turn_id = int(turn_id)
    if turn_id == 1:
        # Q1: 为什么 (WHY) - 挑战 groupby 归类
        return ("Tutor: 你说 groupby 是描述性, ttest 是诊断性. "
                "**为什么** ttest 属于诊断性而非描述性? 描述性和诊断性的本质区别是什么? "
                "(提示: 想想它们回答的业务问题)")
    elif turn_id == 2:
        # Q2: 反例 (COUNTEREXAMPLE)
        return ("Tutor: 你说描述性回答发生了什么, 诊断性回答为什么. "
                "**反例**: 如果我用 groupby 对比 treated vs control 的 re78 均值, "
                "这算不算在回答为什么 treated 组消费更高? 那 groupby 是描述性还是诊断性? "
                "边界在哪?")
    elif turn_id == 3:
        # Q3: 若前提变 (WHAT IF) - 挑战 Cohen's d 忽略
        return ("Tutor: 你报了 p=0.048 就建议放量. **若前提变**: 假设样本量从 445 涨到 445000, "
                "同样的均值差, p 值会变小还是变大? Cohen's d 会变吗? "
                "**凭什么**只看 p 值就决定放量? 你的 d=0.11 是什么意思?")
    elif turn_id == 4:
        # Q4: 凭什么 (ON WHAT GROUND) - 挑战 OLS 协变量
        return ("Tutor: 你说 OLS 是诊断性. **凭什么** OLS 要把 re75 放进协变量? "
                "不放 re75, treat 系数会怎样? 这与 CUPED 的思想有什么相通? "
                "(提示: 混杂偏误 vs 方差缩减)")
    elif turn_id == 5:
        # Q5: 如何 (HOW) - RFM 行动转化
        return ("Tutor: 最后, 你跑了 ttest 和 OLS, 但还没碰 RFM. **如何**用 NSW 数据 "
                "构造 R/F/M 三列? **如何**从 RFM 分群给出差异化营销动作? "
                "Champions 群和 At-Risk 群的动作能一样吗? 为什么?")
    else:
        return "Tutor: 本轮 Socratic loop 结束 (5 问已完). 进入 cell4 记录 student_model."

# 模拟 6 轮 Socratic 对话 (5 问 + 收尾)
print("=" * 60)
print("Socratic Loop (5 轮, 静态 if/else 模拟, 不调 LLM API)")
print("=" * 60)
for t in range(1, 7):
    resp = socratic_turn(t, student_submission)
    print("\n--- Turn %d ---" % t)
    print(resp)
    if t <= 5:
        print("[Student 思考后回答, 进入 Turn %d]" % (t + 1))


In [ ]:
# === cell4: student_model.json 读写 (记录掌握度/盲点) ===
# 6 个 ILO 的 mastery 字段, 0.0-1.0, 阈值 0.8

import os, json

student_model = {
    "unit": "U-E2-D1",
    "student_id": "demo_student",
    "ilo_mastery": {
        "ILO1_framework": 0.6,   # 四层框架
        "ILO2_funnel": 0.5,      # AARRR 漏斗
        "ILO3_RFM": 0.3,         # RFM 分群
        "ILO4_test": 0.4,        # t检验/卡方 (盲点: Cohen d)
        "ILO5_OLS": 0.2,         # OLS (盲点: re75 协变量)
        "ILO6_NSW_mapping": 0.5  # NSW 营销映射
    },
    "blindspots": [
        "统计显著 vs 商业显著 (Cohen d 忽略)",
        "OLS 协变量 re75 与 CUPED 的连接",
        "RFM 五分群 -> 差异化营销动作"
    ],
    "weak_history": [],  # weak_loop 触发记录
    "tutorial_count_today": 1,
    "last_tutorial_date": "2026-07-26"
}

model_path = "student_model.json"
with open(model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

print("Wrote " + model_path)
print(json.dumps(student_model, ensure_ascii=False, indent=2))

# 读取校验
with open(model_path, encoding="utf-8") as f:
    loaded = json.load(f)
print("\nReload OK. blindspots count = " + str(len(loaded["blindspots"])))
below = [k for k, v in loaded["ilo_mastery"].items() if v < 0.8]
print("Below-threshold ILOs: " + str(below))


## Hattie 4 级 Formative Feedback (避免 Self 级表扬)

> John Hattie *Visible Learning*: 反馈聚焦 Task / Process / Self-Regulation / Feed-Forward,
> 不表扬人格 (Self 级 "你真聪明" 无效). 本单元反馈如下:

**[TASK]** (任务级): 你的 ttest_ind 代码正确, p=0.048 计算无误。
但任务要求"同时报告 Cohen's d", 你只报了 p 值, **d=0.11 未被解读**。
回去补 d 的解读: "d<0.2 为小效应, 商业上可能不值得放量"。

**[PROCESS]** (过程级): 你把 OLS 归为诊断性, 方向对, 但**过程缺一步**:
你没解释为何要把 re75 放进协变量。正确过程是: "re75 是预处理变量,
不放则 treat 系数被基线消费差异混杂, 与 CUPED 用预处理信息缩减方差的思想相通"。
重做 practice.md D4 的 Worked-Faded 第一阶段。

**[SELF-REG]** (自我调节级): 你在 RFM 题上跳过了 qcut, 直接用固定阈值。
这是**自我监控缺失**: 你的 student_model.json 显示 ILO3_RFM=0.3,
低于 0.8 阈值, 应主动触发 weak_loop 回退 D5 worked example。
下次遇到分群任务, 先问自己"该用分位数还是固定阈值? 分布是偏态吗?"。

**[FEED-FORWARD]** (前馈级): 下一步 (Day 2 CLV 预测) 会用到今天的 RFM 与 OLS。
你的 ILO5_OLS=0.2 是最大盲点, **进 Day 2 前**必须完成:
(1) practice.md D4 全部 5 次 rep; (2) schedule.json C6 (OLS+CUPED) 卡片复习到 due[3]=8 天;
(3) tutorial.ipynb 重做 Turn 4 的"凭什么放 re75"问答。
否则 Day 2 的 BG/NBD 模型会因混杂变量处理不清而崩盘。

> 注: 无 [SELF] 级表扬 ("你真棒""聪明"等), 因 Hattie 元分析显示 Self 级反馈效应量极低 (d≈0.16)。


## 限频 (防依赖) + Exit Artifact

### 限频
- 本单元 (U-E2-D1) **每天最多 1 次** tutorial, 防止学生依赖 tutor 而非自主 retrieval。
- 触发限频: student_model.json 的 `tutorial_count_today >= 1` 即拒绝新会话, 提示"明日再来"。
- 重置: 每日 0 点 `tutorial_count_today` 清零 (实际由 schedule.json 的 due 数组驱动)。

### Exit Artifact (本单元 tutorial 结束必交)
1. **2-3 盲点清单** (从 student_model.json 的 blindspots 提取, 自己用一句话改写, 不许复制):
   - 盲点 1: ___ (如: 我把 p<0.05 等同于"商业值得放量", 忽略了 Cohen's d)
   - 盲点 2: ___ (如: 我没把 re75 放 OLS 协变量, 不懂 CUPED 连接)
   - 盲点 3: ___ (如: 我用固定阈值做 RFM, 没用 qcut 自适应分布)
2. **推荐复习单元** (按盲点映射):
   - 若盲点含"统计显著≠商业显著" -> 复习 schedule.json C4 (t检验+Cohen's d), due[1]=3 天
   - 若盲点含"OLS 协变量" -> 复习 schedule.json C6 (OLS+CUPED), due[1]=3 天
   - 若盲点含"RFM 分群" -> 复习 schedule.json C3 (RFM), due[1]=3 天
3. **下次 tutorial 前置任务** (自主 retrieval, 不许翻 notes.md):
   - 口述 NSW 数据 treat/re74/re75/re78 的营销含义
   - 口述 AARRR 五阶段与 1 个关键指标
   - 口述"为什么 OLS 要放 re75" (CUPED 连接)

> Exit artifact 提交后, student_model.json 的 blindspots 字段清空, weak_history 追加本次记录。
> mastery 未达 0.8 的 ILO 进入下一轮 interleaving (见 practice.md)。
